## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Email frequency and unsubscribes: Mantel-extension test

**Mantel-Extension Test** · a worked example from the StatNex teaching guide

**The story.** A brand varied how many marketing emails customers received (0, 1, 2-3 or 4+ per week) and tracked who unsubscribed. The results are split by customer type: new customers, returning customers and loyalty members.

**The question.** Do more emails lead to more unsubscribes, within every customer type?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/extended_mantel.csv` (13 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Customer type,Emails per week,Unsubscribed,Stayed,Total
New customers,0 per week,3.0,57.0,60.0
,1 per week,6.0,74.0,80.0
,2-3 per week,9.0,51.0,60.0
,4 or more per week,12.0,38.0,50.0
Returning customers,0 per week,2.0,58.0,60.0
,1 per week,5.0,95.0,100.0
,2-3 per week,9.0,71.0,80.0
,4 or more per week,11.0,44.0,55.0
Loyalty members,0 per week,1.0,39.0,40.0
,1 per week,2.0,78.0,80.0
,2-3 per week,4.0,76.0,80.0
,4 or more per week,6.0,54.0,60.0
,Total,70.0,735.0,805.0
""")

## Helper: cochran_mantel_haenszel

A step that no library does, written out. In the code download it is the file `cochran_mantel_haenszel.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Stratified-table statistics that SciPy and statsmodels do not give in the app's form.

cmh_correlation   the CMH correlation statistic, rows and columns scored 1, 2, 3, ...
generalised_cmh   Landis, Heyman and Koch's three statistics for r x c strata
mh_risk_ratio     the Mantel-Haenszel common risk ratio and the SE of its log
logit_pooled      the inverse-variance (Woolf, "logit") pooled ratio and the SE of its log

Used by the Cochran-Mantel-Haenszel test (2 x 2 strata) and the extended Mantel test
(k x 2 strata).
"""

import numpy as np
from scipy import stats


def cmh_correlation(tables):
    """CMH correlation statistic and its P value (chi-square, 1 df).

    Rows and columns are scored 1, 2, 3, ... in table order (SAS PROC FREQ's default, Mantel's equally spaced scores),
    with the hypergeometric variance on n - 1. On one table the statistic is
    (n - 1) r^2, the Mantel-Haenszel chi-square; on 2 x 2 strata it is the textbook CMH
    statistic (statsmodels' test_null_odds(correction=False), R's mantelhaen.test).
    """
    numerator = variance = 0.0
    for table in tables:
        n = table.sum()
        if n < 2:
            continue
        row_totals, col_totals = table.sum(axis=1), table.sum(axis=0)
        x = np.arange(1.0, len(row_totals) + 1)
        y = np.arange(1.0, len(col_totals) + 1)
        observed = (np.outer(x, y) * table).sum()
        expected = (x * row_totals).sum() * (y * col_totals).sum() / n
        var_x = (x**2 * row_totals).sum() - (x * row_totals).sum() ** 2 / n
        var_y = (y**2 * col_totals).sum() - (y * col_totals).sum() ** 2 / n
        numerator += observed - expected
        variance += var_x * var_y / (n - 1)
    statistic = numerator**2 / variance
    return statistic, stats.chi2.sf(statistic, 1)


def mh_risk_ratio(tables):
    """Mantel-Haenszel common risk ratio of column 1 (row 1 over row 2), and SE(log).

    The variance is Greenland & Robins (1985). statsmodels' StratifiedTable has the
    estimate (riskratio_pooled) but no interval.
    """
    a, b, c, d = (np.array([t[i, j] for t in tables], float)
                  for i, j in ((0, 0), (0, 1), (1, 0), (1, 1)))
    n = a + b + c + d
    row1, row2 = a + b, c + d
    numerator, denominator = (a * row2 / n).sum(), (c * row1 / n).sum()
    estimate = numerator / denominator if denominator > 0 else np.nan
    if numerator > 0 and denominator > 0:
        variance = ((row1 * row2 * (a + c) - a * c * n) / n**2).sum() / (numerator * denominator)
        return estimate, np.sqrt(variance)
    return estimate, np.nan


def logit_pooled(estimates, variances):
    """Inverse-variance pooled ratio exp(sum w log r / sum w), w = 1 / Var(log r), and SE(log).

    A stratum with an empty cell has no finite log ratio or variance, so it is left out;
    with none left the estimator is undefined -- which is why the Mantel-Haenszel form,
    a ratio of sums, is reported beside it.
    """
    usable = (np.isfinite(estimates) & (estimates > 0)
              & np.isfinite(variances) & (variances > 0))
    if not usable.any():
        return np.nan, np.nan
    weights = 1 / variances[usable]
    estimate = np.exp((weights * np.log(estimates[usable])).sum() / weights.sum())
    return estimate, 1 / np.sqrt(weights.sum())


def generalised_cmh(tables):
    """[(name, df, statistic, P)] for strata larger than 2 x 2 (Landis, Heyman & Koch 1978).

    Each statistic is G' V^-1 G, with G the sum over strata of B (n - m) -- observed
    minus expected counts under conditional independence, the table stacked column by
    column -- and V the sum of B Cov B', Cov = N^2/(N-1) (V_col (x) V_row). They differ
    only in B: every cell but the last row and column (general association), each
    row's sum of column scores (row mean scores), or the score product (correlation).
    The scores are cmh_correlation's table scores, so the last statistic is
    that one.
    """
    rows, cols = tables[0].shape
    keep_rows, keep_cols = np.eye(rows)[:-1], np.eye(cols)[:-1]
    names = ("General association", "Row mean scores differ", "Correlation")
    sums, variances = [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]
    for table in tables:
        n = table.sum()
        if n < 2:
            continue
        row_totals, col_totals = table.sum(axis=1), table.sum(axis=0)
        p_row, p_col = row_totals / n, col_totals / n
        covariance = n**2 / (n - 1) * np.kron(np.diag(p_col) - np.outer(p_col, p_col),
                                              np.diag(p_row) - np.outer(p_row, p_row))
        gap = (table - np.outer(row_totals, col_totals) / n).flatten(order="F")
        x = np.arange(1.0, len(row_totals) + 1)
        y = np.arange(1.0, len(col_totals) + 1)
        for k, contrast in enumerate((np.kron(keep_cols, keep_rows),
                                      np.kron(y[None, :], keep_rows),
                                      np.kron(y[None, :], x[None, :]))):
            sums[k] = sums[k] + contrast @ gap
            variances[k] = variances[k] + contrast @ covariance @ contrast.T
    out = []
    for name, total, variance in zip(names, sums, variances):
        variance = np.atleast_2d(variance)
        df = np.linalg.matrix_rank(variance)
        statistic = np.atleast_1d(total) @ np.linalg.pinv(variance) @ np.atleast_1d(total)
        out.append((name, df, statistic, stats.chi2.sf(statistic, df)))
    return out

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
# cmh_correlation: defined in the helper cell cochran_mantel_haenszel above (cochran_mantel_haenszel.py in the code download)

pd.set_option("display.width", 120)

## The data

In [ ]:
# One k x 2 table per stratum, stacked: the rows of a stratum are the ordered levels
# (doses), in the order they rise; the columns are cases / controls.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# A stratum's name is printed on its first row only; carry it down to the rows below.
stratum = data["Customer type"].ffill()
labels = data["Emails per week"]
# Rows and columns named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) are margins, not data.
# The app always leaves them out of a stratified table, ticked or not. Spaces,
# underscores, case and full-width letters are ignored when names are compared.
# is_total: defined in the helper cell totals above (totals.py in the code download)

columns = [name for name in ["Unsubscribed", "Stayed"] if not is_total(name)]
keep = [not is_total(label) for label in labels]
kept = data.loc[keep, columns].astype(float).set_axis(labels[keep])
strata = {name: table for name, table in kept.groupby(stratum[keep].to_numpy(), sort=False)}
# A stratum's rows are matched to the other strata's by their place, and named from the
# first stratum's, as the app names them.
row_names = next(iter(strata.values())).index
strata = {name: table.set_axis(row_names) for name, table in strata.items()}
tables = [table.to_numpy() for table in strata.values()]

## Observed frequency

In [ ]:
for name, table in strata.items():
    print(f"\n{name}")
    print(table)

## Test of independence within each stratum

In [ ]:
# The chi-square asks whether the levels differ at all (k - 1 degrees of freedom); the
# Mantel-Haenszel chi-square asks only whether the proportion moves one way along the
# levels (1 degree of freedom). It is the correlation statistic on one table,
# (n - 1) r^2, with the levels scored 1, 2, 3, ... in order.
stratum_tests = []
for name, table in strata.items():
    counts = table.to_numpy()
    pearson = stats.chi2_contingency(counts, correction=False)
    likelihood = stats.chi2_contingency(counts, correction=False, lambda_="log-likelihood")
    mantel_haenszel, mantel_haenszel_p = cmh_correlation([counts])
    cramers_v = np.sqrt(pearson.statistic / (counts.sum() * (min(counts.shape) - 1)))
    tests = pd.DataFrame(
        [["Chi-square", pearson.dof, pearson.statistic, pearson.pvalue],
         ["Likelihood-ratio chi-square", likelihood.dof, likelihood.statistic, likelihood.pvalue],
         ["Mantel-Haenszel chi-square", 1, mantel_haenszel, mantel_haenszel_p],
         ["Cramer's V", np.nan, cramers_v, np.nan]],
        columns=["test", "df", "value", "P"])
    stratum_tests.append(tests)
    print(f"\n{name}")
    print(tests.to_string(index=False))

## Extended Mantel test (conditional independence)

In [ ]:
# The trend measured inside every stratum and only then added up, so a confounder that
# differs between strata cannot manufacture a trend. The levels and the two columns are
# scored 1, 2, 3, ... in order, Mantel's (1963) equally spaced scores and SAS's default
# (cochran_mantel_haenszel.py). Neither SciPy nor statsmodels has this statistic.
cmh, cmh_p = cmh_correlation(tables)
conditional = pd.DataFrame([["CMH correlation statistic", 1, cmh, cmh_p]],
                           columns=["test", "df", "value", "P"])
print()
print(conditional.to_string(index=False))

## What the result says

The Mantel-extension test asks whether a rate rises steadily across ordered levels, after it splits the data into groups. Here the groups are the three customer types. The combined statistic is 26.29 with 1 degree of freedom, and the P value is far below 0.01. A P value is the chance of a trend at least this strong if email frequency made no difference.

The trend appears in each group. The Mantel-Haenszel chi-square is 10.91 for new customers, 11.58 for returning customers and 3.93 for loyalty members (P = 0.047, the weakest). New customers lose the most: 3 of 60 unsubscribe with no emails, but 12 of 50 with 4 or more per week.

So more emails go with more unsubscribes in every customer type.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.